# Qwen3-14B base evaluation


In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv>=1,<2"], check=True)
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get("CRASHDIAG_ENV_FILE", LAUNCH_DIR / ".env")).expanduser()
if not ENV_FILE.is_absolute():
    ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if not ENV_FILE.is_file():
    raise RuntimeError(f"CrashDiag .env not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=True)
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path(os.environ.get("CRASHDIAG_WORKDIR", LAUNCH_DIR / "CrashDiag-runtime")).expanduser().resolve()
if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f"CRASHDIAG_WORKDIR exists and is not a Git checkout: {WORKDIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print(f"env_file={ENV_FILE}")
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BASE_MODEL = "Qwen/Qwen3-14B"
MODEL_SLUG = "qwen3_14b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{MODEL_SLUG}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"base_model={BASE_MODEL}")
print(f"dataset_run_id={DATASET_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

BASE_RUN_ID = os.environ.get("CRASHDIAG_BASE_QWEN3_14B_RUN_ID") or ist_run_id("base-eval")
DATASET_DIR = Path("artifacts/datasets")
CURRICULUM = os.environ.get("CRASHDIAG_CURRICULUM", "hard-v3").strip().lower()
EVAL_FILE = "grpo_hard_eval.jsonl" if CURRICULUM == "hard-v3" else "grpo_eval.jsonl"
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("datasets", DATASET_DIR)
assert (DATASET_DIR / EVAL_FILE).is_file(), f"dataset stage missing {EVAL_FILE}; check CRASHDIAG_DATASET_RUN_ID={DATASET_RUN_ID}"
print(f"BASE_RUN_ID={BASE_RUN_ID}")
print(f"curriculum={CURRICULUM}")
print(f"eval_file={EVAL_FILE}")


In [ ]:
from training.evaluate_jsonl import main as evaluate_main

exit_code = evaluate_main([
    "--model", BASE_MODEL,
    "--dataset", str(DATASET_DIR / EVAL_FILE),
    "--output-dir", "outputs/base-eval",
    "--load-in-4bit",
    "--precision", "bf16",
    "--max-new-tokens", "64",
    "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
    "--artifact-bucket", BUCKET_ID,
    "--run-id", BASE_RUN_ID,
    "--artifact-stage", "base-eval",
])
if exit_code: raise RuntimeError(f"base evaluation failed: {exit_code}")


In [ ]:
from IPython.display import SVG, display

REPORTS_DIR = Path("outputs/base-eval") / "reports"
charts = sorted(REPORTS_DIR.glob("*.svg"))
if not charts:
    raise RuntimeError(f"No evaluation SVG charts were generated in {REPORTS_DIR}")
print(f"Uploaded evaluation reports: hf://buckets/{BUCKET_ID}/runs/{BASE_RUN_ID}/base-eval/reports")
for chart in charts:
    display(SVG(filename=str(chart)))
